# Build a ReAct Agent with LangGraph on Modal

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/starter-examples/langgraph-react-agent/tutorial_langgraph_react_agent.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Build a simple ReAct (Reason + Act) agent using LangGraph's prebuilt `create_react_agent` and run it on [Modal](https://modal.com).

**What you'll learn:**
- Define tools with `@tool` and wire them into a LangGraph agent
- Package the agent as a `modal.App` function with an inline `modal.Image`
- Inject the OpenAI key with a `modal.Secret` and run the agent with `modal run`

**Pattern:**
```
User: "What is 12 * 7 plus 3?"
  → Agent reasons: need to multiply first
  → Calls multiply(12, 7) → 84
  → Reasons: now add 3
  → Calls add(84, 3) → 87
  → Returns: "87"
```

---

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/starter-examples/langgraph-react-agent/
    !pip install -r requirements.txt

from utils.file_viewer import view_file

## Dependencies

The only local dependency is `modal` — LangGraph, LangChain, and OpenAI are installed *inside* the Modal container image, defined in the script:

In [ ]:
view_file("requirements.txt")

## Connect to Modal

Authenticate once. This opens a browser to link your Modal account (sign up free at [modal.com](https://modal.com)):

In [ ]:
!modal setup

## Set your API key

The agent uses OpenAI. The Modal function reads `OPENAI_API_KEY` from a **Modal secret** named `openai-secret`, so the key is available wherever the function runs — not just on your laptop.

Enter your key below:

In [ ]:
import os
from getpass import getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

Store it as a Modal secret so the function can read it when it runs in the cloud (`--force` overwrites an existing secret of the same name):

In [ ]:
!modal secret create openai-secret OPENAI_API_KEY=$OPENAI_API_KEY --force

---

## Code Walkthrough

The entire agent fits in a single file. Let's walk through it:

In [ ]:
view_file("langgraph_react_agent.py")

### Key Components

**1. Modal image & app** — The image declares the dependencies; the app groups the functions. No cluster to provision.

```python
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "langchain-core", "langgraph", "langchain-openai"
)
app = modal.App("langgraph-react-agent", image=image)
```

**2. Tools** — Simple Python functions decorated with `@tool` (LangChain). Must be `async`.

```python
@tool
async def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b
```

**3. Agent function** — Uses LangGraph's `create_react_agent` to wire up the LLM + tools into a ReAct loop, wrapped as a Modal function with the OpenAI secret attached.

```python
@app.function(secrets=[modal.Secret.from_name("openai-secret")])
async def agent(request: str) -> str:
    llm = ChatOpenAI(model="gpt-4o-mini")
    react_agent = create_react_agent(llm, [add, multiply])
    result = await react_agent.ainvoke(...)
```

**4. Local entrypoint** — `modal run` calls this; `agent.remote(...)` runs the function in the cloud and returns the result locally.

```python
@app.local_entrypoint()
def main(request: str = "What is 12 * 7 plus 3?"):
    print(agent.remote(request))
```

---

## Run the Agent

`modal run` builds the image (first run only), executes the function in the cloud, and prints the result. The `--request` flag maps to the `main` entrypoint argument.

In [ ]:
!modal run langgraph_react_agent.py --request "What is 12 * 7 plus 3?"

### Try different prompts

In [ ]:
!modal run langgraph_react_agent.py --request "Multiply 15 by 4 and then add 20"

In [ ]:
!modal run langgraph_react_agent.py --request "What is 100 divided by 4 times 3?"

### Inspect runs in the dashboard

Every `modal run` streams logs to your terminal and records the run in the [Modal dashboard](https://modal.com/apps), where you can browse past executions, logs, and container metrics per function. Open it with:

```bash
modal app list
```

or visit [modal.com/apps](https://modal.com/apps) directly.

---

## Key Takeaways

- **LangGraph + Modal** — Use LangGraph for agent logic, Modal for packaging, scaling, and running it in the cloud
- **Inline image** — Dependencies live in the `modal.Image`, so the only local requirement is `modal`
- **`modal.Secret`** — Injects `OPENAI_API_KEY` into the function wherever it runs
- **`create_react_agent`** — LangGraph's prebuilt ReAct loop handles the reasoning cycle for you
- **Single file** — The entire agent is self-contained in one `.py` file

## Next Steps

- Add more tools (web search, file I/O, database queries)
- Try different LLMs (`gpt-4o`, or Claude via `langchain-anthropic`)
- Attach a GPU to the function to run an open-weights model
- Check out the [multi-agent tutorials](../../multi-agent-workflows/) for more complex patterns